<a href="https://colab.research.google.com/github/Sarztak/WindyGraph-A-graph-based-recommender-for-restaurants-in-Chicago/blob/main/Factorization_Model_Machine_Learning_II.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import files

uploaded = files.upload()


Saving processed_review_data.pkl to processed_review_data.pkl


In [3]:
from google.colab import files

uploaded = files.upload()


Saving processed_restaurant_data.pkl to processed_restaurant_data.pkl


In [4]:
!pip install lightfm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 5.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for lightfm: filename=lightfm-1.17-cp311-cp311-linux_x86_64.whl size=831163 sha256=43587c53e9e6b1811e38871931197977049bac98f1fbd945e11ee4cefc6f2efe
  Stored in directory: /root/.cache/pip/wheels/b9/0d/8a/0729d2e6e3ca2a898ba55201f905da7db3f838a33df5b3fcdd
Successfully built lightfm


In [5]:
import pandas as pd

review_restaurantdata = pd.read_pickle("processed_review_data.pkl")
restaurant_data = pd.read_pickle("processed_restaurant_data.pkl")


In [6]:
# Merge on restaurant_id (review_restaurantdata) and id (restaurant_data)
merge_reviewndata = review_restaurantdata.merge(restaurant_data[['id', 'categories_list']], left_on='restaurant_id', right_on='id')

# Prepare interaction dataset: user_id, restaurant_id, rating, categories_list
merge_reviewndata = merge_reviewndata[['user_id', 'restaurant_id', 'rating', 'categories_list']].dropna()

merge_reviewndata['categories_list'] = merge_reviewndata['categories_list']

# Faster training
merge_reviewndata = merge_reviewndata.sample(n=5000, random_state=123)


In [8]:
# Set up LightFM dataset
from lightfm.data import Dataset
dataset = Dataset()
dataset.fit(users=merge_reviewndata['user_id'], items=merge_reviewndata['restaurant_id'],
            item_features=set(cat for cats in merge_reviewndata['categories_list'] for cat in cats))

(interactions, weights) = dataset.build_interactions([
    (row['user_id'], row['restaurant_id'], row['rating']) for _, row in merge_reviewndata.iterrows()
])

item_features = dataset.build_item_features([
    (row['restaurant_id'], row['categories_list']) for _, row in merge_reviewndata.iterrows()
])

In [10]:
# Train the LightFM model
from lightfm import LightFM
model = LightFM(loss='warp', no_components=30)
model.fit(interactions, item_features=item_features, epochs=10, num_threads=4)


In [11]:
import pickle
import joblib

with open("lightfm_model.pkl", "wb") as f:
    pickle.dump(model, f)

joblib.dump(dataset, "lightfm_dataset.joblib")
joblib.dump(interactions, "interactions_matrix.joblib")
joblib.dump(item_features, "item_features_matrix.joblib")

print("files saved.")


files saved.


In [12]:
import pickle
import joblib
import numpy as np
from lightfm import LightFM
from lightfm.data import Dataset

# Load saved model and components
with open("lightfm_model.pkl", "rb") as f:
    model = pickle.load(f)

dataset = joblib.load("lightfm_dataset.joblib")
interactions = joblib.load("interactions_matrix.joblib")
item_features = joblib.load("item_features_matrix.joblib")

user_id_map = dataset.mapping()[0]
item_id_map = dataset.mapping()[2]

# Inverse maps
id_to_user = {v: k for k, v in user_id_map.items()}
id_to_item = {v: k for k, v in item_id_map.items()}

def recommend_known_user(user_id_str, model, dataset, interactions, item_features, N=5):
    user_id_map = dataset.mapping()[0]
    item_id_map = dataset.mapping()[2]
    item_index_to_id = {v: k for k, v in item_id_map.items()}

    try:
        user_idx = user_id_map[user_id_str]
    except KeyError:
        print(f"User ID '{user_id_str}' not found.")
        return []

    n_items = interactions.shape[1]
    scores = model.predict(user_ids=user_idx,
                           item_ids=np.arange(n_items),
                           item_features=item_features)

    known_items = interactions.tocsr()[user_idx].indices
    scores[known_items] = -np.inf

    top_items = np.argsort(-scores)[:N]
    return [item_index_to_id[i] for i in top_items]


In [14]:
merge_reviewndata['user_id'].unique()[:10]


array(['thV5RSlc8l4QL6Jf4--PIw', 'VqjxVozyfl8-BVNZ5g3D8Q',
       'RiB1mlLFirfWDqjTv09eAA', 'tH_RQBcxqYAmB-A4RVLHaw',
       'Sw-IGWHmnXji2z43x4A5OA', 'Y_NVgxBRcII68dSuwfNHAQ',
       'ukBY6FqUM0NCf7C1w8am1w', 'wrGhtevrzlMRcbGo72Aubg',
       '3xUa12mLDyWAb3ht5iVW1w', '37RrURW8yCJoo1DEgdiiYg'], dtype=object)

In [19]:

# Mapping from restaurant ID to name
id_to_name = dict(zip(restaurant_data['id'], restaurant_data['name']))

# Recommendations with names
print(f"Top recommendations for user '{user}':")
for item_id in recommended_items:
    name = id_to_name.get(item_id, "[Unknown]")
    print(f"- {name} (ID: {item_id})")


Top recommendations for user 'ukBY6FqUM0NCf7C1w8am1w':
- Bill's Place Restaurant (ID: uWo53qPJLoHQHh76IWg-XQ)
- Original Pancake House (ID: FM1y-KHfhdSp_d74ryNG3Q)
- Brothers Restaurant (ID: 9nSQL9pl0UMvWMORoMRtAA)
- Stax Cafe (ID: jpdKZ8XBoRHrtDiB887o9A)
- Sunshine Restaurant (ID: CTbOVEQFhq8pixZgr7SiWA)
